# Project 07 — BROKEN notebook (debugging exercise)

This notebook fits a **Normal** likelihood to outlier-contaminated data and lets a single outlier dominate. Run it, see the dragged line and inflated sigma, find each bug, and fix it. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
x, y, out = data['x'], data['y'], data['is_outlier']

### Model — BUG 1: a Normal likelihood lets gross outliers dominate.

In [ ]:
# BUG 1: Normal noise penalizes far points QUADRATICALLY, so a handful of
#        outliers dominate the fit. The slope tilts and sigma inflates to
#        'explain' the outliers as ordinary noise.
with pm.Model() as model:
    alpha = pm.Normal('alpha', 0.0, 5.0)
    beta = pm.Normal('beta', 0.0, 5.0)
    sigma = pm.HalfNormal('sigma', 5.0)
    mu = alpha + beta * x
    pm.Normal('y', mu=mu, sigma=sigma, observed=y)
    idata = pm.sample(draws=800, tune=800, chains=2, random_seed=RNG,
                      progressbar=False, idata_kwargs={'log_likelihood': True})

In [ ]:
# Converges fine -> the trap. Note the inflated sigma and the off-truth slope.
print(az.summary(idata, var_names=['alpha', 'beta', 'sigma']))
print('true beta =', data['truth']['beta'], 'true sigma =', data['truth']['sigma'])

### A 'robust' attempt — BUG 2: nu is fixed FAR too large, so the Student-t is Normal in disguise and stays non-robust.

In [ ]:
# BUG 2: using a Student-t but PINNING nu=100. At nu=100 the t is
#        indistinguishable from a Normal, so this 'robust' model is not
#        robust at all. nu must be a free parameter (or set small) to let
#        the heavy tail absorb the outliers.
with pm.Model() as model_t:
    alpha = pm.Normal('alpha', 0.0, 5.0)
    beta = pm.Normal('beta', 0.0, 5.0)
    sigma = pm.HalfNormal('sigma', 5.0)
    mu = alpha + beta * x
    pm.StudentT('y', nu=100.0, mu=mu, sigma=sigma, observed=y)  # BUG 2
    idata_t = pm.sample(draws=800, tune=800, chains=2, random_seed=RNG,
                        progressbar=False)
print(az.summary(idata_t, var_names=['alpha', 'beta', 'sigma']))
# Still dragged: nu=100 defeats the purpose. Fix: nu = pm.Gamma('nu', 2, 0.1).